# Part 2: Tool Usage — SOC Edition

In [1]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

# Auto-detect pre-provisioned environment
config_path = Path.cwd().parent / "workshop_config.json"
if not config_path.exists():
    config_path = Path.cwd() / "workshop_config.json"
if config_path.exists():
    _cfg = json.loads(config_path.read_text())
    PROJECT_ENDPOINT = _cfg["PROJECT_ENDPOINT"]
    MODEL_DEPLOYMENT_NAME = _cfg["MODEL_DEPLOYMENT_NAME"]
else:
    raise FileNotFoundError(
        "workshop_config.json not found. Run 00-setup.ipynb first."
    )

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Connected to: https://fndryiac2ttkx3.services.ai.azure.com/api/projects/iac-ops-demo
   Model: gpt-4.1-mini


---
## Section 2: Tool Usage (~10 min)

Agents become powerful when you give them **tools**. Foundry supports:

| Tool Type | Description | SOC Use Case |
|-----------|-------------|----------|
| **Function Calling** | Call your custom functions | IP reputation lookups, alert queries, user context retrieval |
| **Web Search** | Search the web in real-time | CVE research, threat intelligence, MITRE ATT&CK lookups |
| **File Search** | Search uploaded documents (RAG) | SOC playbooks, triage procedures, compliance docs |
| **Code Interpreter** | Run Python code | Alert statistics, incident metrics, trend analysis |

In [2]:
# --- 2.1 Function Calling: IP Reputation Lookup ---
# Define a Python function, register it as a tool schema, and handle calls via the Responses API.

# The function we want the agent to call
def lookup_ip_reputation(ip_address: str) -> str:
    """Look up the threat intelligence reputation for a given IP address."""
    ip_intel = {
        "203.0.113.42": json.dumps({"IP": "203.0.113.42", "Reputation": "Clean", "ISP": "Contoso Corp", "Country": "US", "IsTor": False, "IsProxy": False, "IsVPN": False, "ThreatCategory": "None"}),
        "198.51.100.7": json.dumps({"IP": "198.51.100.7", "Reputation": "Malicious", "ISP": "BulletproofHost Ltd", "Country": "NG", "IsTor": False, "IsProxy": True, "IsVPN": True, "ThreatCategory": "C2/Proxy"}),
        "198.51.100.99": json.dumps({"IP": "198.51.100.99", "Reputation": "Malicious", "ISP": "ShadowNet", "Country": "RU", "IsTor": True, "IsProxy": True, "IsVPN": False, "ThreatCategory": "Tor Exit Node"}),
        "192.0.2.10": json.dumps({"IP": "192.0.2.10", "Reputation": "Clean", "ISP": "Contoso Corp", "Country": "US", "IsTor": False, "IsProxy": False, "IsVPN": False, "ThreatCategory": "None"}),
        "192.0.2.55": json.dumps({"IP": "192.0.2.55", "Reputation": "Clean", "ISP": "Contoso UK Ltd", "Country": "UK", "IsTor": False, "IsProxy": False, "IsVPN": False, "ThreatCategory": "None"}),
    }
    return ip_intel.get(ip_address, json.dumps({"IP": ip_address, "Reputation": "Unknown", "Message": "IP not found in threat intelligence database"}))

# Define the tool schema for the agent
ip_tool = FunctionTool(
    name="lookup_ip_reputation",
    description="Look up the threat intelligence reputation for a given IP address, including ISP, country, Tor/VPN/proxy flags, and threat category.",
    parameters={
        "type": "object",
        "properties": {
            "ip_address": {
                "type": "string",
                "description": "The IP address to look up, e.g. '198.51.100.7'",
            }
        },
        "required": ["ip_address"],
        "additionalProperties": False,
    },
    strict=True,
)

ip_agent = project_client.agents.create_version(
    agent_name="02-soc-ip-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a SOC threat intelligence analyst. Use the lookup_ip_reputation tool "
            "to check IP addresses against the threat intelligence database. "
            "Summarize the findings and highlight any malicious indicators (Tor, VPN, proxy, known C2)."
        ),
        tools=[ip_tool],
    ),
)

# Send a question — the agent may request a tool call
response = openai_client.responses.create(
    input="Check the reputation of IP 198.51.100.7 — it appeared in an impossible travel alert.",
    extra_body={"agent_reference": {"name": ip_agent.name, "type": "agent_reference"}},
)

# Check if the agent made a function call
tool_calls = [item for item in response.output if item.type == "function_call"]
if tool_calls:
    call = tool_calls[0]
    args = json.loads(call.arguments)
    print(f"🔧 Agent called: {call.name}({args})")
    result = lookup_ip_reputation(**args)
    print(f"   Result: {result}")

    # Send the tool result back to get the final answer
    response = openai_client.responses.create(
        input=[{
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": result,
        }],
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": ip_agent.name, "type": "agent_reference"}},
    )

print(f"\n🧑 Check the reputation of IP 198.51.100.7")
print(f"🤖 {response.output_text}")
print(f"\n✅ Function tool agent: {ip_agent.name} v{ip_agent.version}")

🔧 Agent called: lookup_ip_reputation({'ip_address': '198.51.100.7'})
   Result: {"IP": "198.51.100.7", "Reputation": "Malicious", "ISP": "BulletproofHost Ltd", "Country": "NG", "IsTor": false, "IsProxy": true, "IsVPN": true, "ThreatCategory": "C2/Proxy"}

🧑 Check the reputation of IP 198.51.100.7
🤖 The IP address 198.51.100.7 is flagged as malicious. It is associated with the ISP BulletproofHost Ltd in Nigeria (NG). This IP is marked as both a VPN and a proxy and falls under the threat category of C2 (Command and Control) or Proxy. These indicators strongly suggest a potential threat actor infrastructure, which aligns with the impossible travel alert. Further investigation and possible blocking of this IP are recommended.

✅ Function tool agent: 02-soc-ip-analyst v1


In [3]:
# --- 2.2 Web Search Tool: Threat Intelligence Research ---
# The built-in WebSearchPreviewTool lets agents search the web — no extra resources needed.

web_agent = project_client.agents.create_version(
    agent_name="02-soc-threat-researcher",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a SOC threat intelligence researcher. Search the web to find current "
            "information about CVEs, threat actors, MITRE ATT&CK techniques, and emerging "
            "cybersecurity threats. Always cite your sources with URLs."
        ),
        tools=[WebSearchPreviewTool()],
    ),
)

response = openai_client.responses.create(
    input="What are the latest techniques used in MFA fatigue attacks in 2025-2026, and what mitigations does Microsoft recommend?",
    extra_body={"agent_reference": {"name": web_agent.name, "type": "agent_reference"}},
)
print("🧑 What are the latest MFA fatigue attack techniques and mitigations?\n")
print(f"🤖 {response.output_text}")
print(f"\n✅ Web search agent: {web_agent.name} v{web_agent.version}")

🧑 What are the latest MFA fatigue attack techniques and mitigations?

🤖 The latest techniques used in MFA fatigue (or MFA bombing) attacks in 2025-2026 rely on exploiting human factors by repeatedly sending MFA push notifications to users until they approve a malicious login request out of frustration, distraction, or fatigue. Attackers typically gain stolen credentials (username and password) from phishing, leaks, or breaches, then flood users with authentication requests to wear them down. The attack works because push-based MFA often lacks meaningful context, rate limiting, and relies heavily on user judgment, which can be compromised under stress.

Key tactics in these attacks include:
- Repeated automated MFA push notifications triggering user fatigue.
- Using stolen credentials for login attempts from varied locations or times.
- Exploiting low friction of push MFA (just a tap approves).
- Social engineering to exploit user distraction or off-hours vulnerability.

Microsoft's rec

In [4]:
# --- 2.3 File Search Tool: SOC Playbook RAG ---
# Upload our SOC alert triage spec to a vector store, then let the agent search it.

# Create vector store and upload the SOC spec
vector_store = openai_client.vector_stores.create(name="SOC-Playbooks")
spec_path = Path(__file__).parent / "sample_data" / "soc_alert_spec.md" if '__file__' in dir() else Path.cwd() / "sample_data" / "soc_alert_spec.md"

with spec_path.open("rb") as f:
    openai_client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id, file=f
    )
print(f"📄 Uploaded {spec_path.name} to vector store '{vector_store.name}'")

# Create agent with file search
doc_agent = project_client.agents.create_version(
    agent_name="02-soc-playbook-expert",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a SOC playbook expert. Use file search to answer questions "
            "about alert triage procedures, severity SLAs, and investigation workflows. "
            "Cite specific data from the SOC documentation."
        ),
        tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    ),
)

response = openai_client.responses.create(
    input="What is the SLA for acknowledging a Critical severity alert, and what is the escalation path?",
    extra_body={"agent_reference": {"name": doc_agent.name, "type": "agent_reference"}},
)
print(f"\n🧑 What is the SLA for Critical severity alerts?\n")
print(f"🤖 {response.output_text}")
print(f"\n✅ File search agent: {doc_agent.name} v{doc_agent.version}")
print(f"   Vector store ID: {vector_store.id}")

📄 Uploaded soc_alert_spec.md to vector store 'SOC-Playbooks'

🧑 What is the SLA for Critical severity alerts?

🤖 The SLA for acknowledging a Critical severity alert is 5 minutes. The escalation path for a Critical alert is an immediate page to the Level 3 (L3) analyst plus the Chief Information Security Officer (CISO).

This means when a Critical alert is generated, it must be acknowledged within 5 minutes, and it is escalated immediately by paging the senior analyst (L3) and the CISO for urgent attention.

Reference from the SOC Alert Triage documentation:

| Level    | SLA (Acknowledge) | SLA (Resolve) | Escalation Path              |
|----------|--------------------|---------------|-----------------------------|
| Critical | 5 min              | 30 min        | Immediate page to L3 + CISO  | 

 and 

✅ File search agent: 02-soc-playbook-expert v1
   Vector store ID: vs_BOGramajSOpv3zhg1OSep6CF


In [5]:
# --- 2.4 Code Interpreter: SOC Alert Analytics ---
# The built-in Code Interpreter lets agents write and run Python code in a sandbox.
# Perfect for analyzing SOC alert data — no custom function tool needed.

from azure.ai.projects.models import CodeInterpreterTool, AutoCodeInterpreterToolParam

# Upload the SOC alerts CSV file for the code interpreter to process
alerts_path = Path.cwd() / "sample_data" / "soc_alerts_data.csv"
alerts_file = openai_client.files.create(purpose="assistants", file=open(alerts_path, "rb"))
print(f"📊 Uploaded {alerts_path.name} (file ID: {alerts_file.id})")

code_agent = project_client.agents.create_version(
    agent_name="02-soc-data-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a SOC data analyst. Use Code Interpreter to analyze the uploaded CSV file "
            "containing SOC alert data. Write Python code to answer questions about alert trends, "
            "response times, and incident patterns. Always show the code you ran and the results. "
            "Include specific numbers and percentages in your answers."
        ),
        tools=[CodeInterpreterTool(container=AutoCodeInterpreterToolParam(file_ids=[alerts_file.id]))],
    ),
)

question = "Which alert type has the most incidents, and what's the average time-to-resolve for each alert type? Show the breakdown."
response = openai_client.responses.create(
    input=question,
    extra_body={"agent_reference": {"name": code_agent.name, "type": "agent_reference"}},
)

# Extract and display any code the agent ran
for item in response.output:
    if item.type == "code_interpreter_call":
        print(f"🔧 Code Interpreter ran:\n{item.code}\n")

print(f"🧑 {question}\n")
print(f"🤖 {response.output_text}")
print(f"\n✅ Code Interpreter agent: {code_agent.name} v{code_agent.version}")

📊 Uploaded soc_alerts_data.csv (file ID: assistant-Dj3KDcBMKpgDB7HBNykQUF)
🔧 Code Interpreter ran:
import pandas as pd

# Load the CSV file to see its contents
file_path = '/mnt/data/assistant-Dj3KDcBMKpgDB7HBNykQUF-soc_alerts_data.csv'
data = pd.read_csv(file_path)

# Display the first few rows to understand the structure
data.head()

🔧 Code Interpreter ran:
# Count the number of incidents by alert type
alert_counts = data['alert_type'].value_counts()

# Calculate the average time to resolve for each alert type (ignoring NaN values)
avg_time_to_resolve = data.groupby('alert_type')['time_to_resolve_min'].mean()

# Combine the results into a DataFrame for clearer presentation
alert_summary = pd.DataFrame({
    'incident_count': alert_counts,
    'average_time_to_resolve_min': avg_time_to_resolve
}).reset_index().rename(columns={'index': 'alert_type'})

alert_summary

🧑 Which alert type has the most incidents, and what's the average time-to-resolve for each alert type? Show the breakdown

---
Proceed to `03-prompts-eval.ipynb`.